# 05 — Launch the BrainTumorLab web app in Colab

Pages: Dashboard · MRI Analysis · AI Segmentation · 3D Visualization · Radiomics · MRI Generation ·
Model Comparison & Reports. A GPU runtime makes segmentation and generation much faster.

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
import os
# use the classifier trained in notebook 02 when it exists
CLF = paths.base / "models" / "classification" / "cnn3d_grade_profile.pt"
if CLF.exists():
    os.environ["BTL_CLASSIFIER"] = str(CLF)
os.environ["BTL_DATA_ROOT"] = str(DATA_ROOT)
from braintumorlab.webapp.server import launch_in_colab
server = launch_in_colab(port=8000)      # click the "Open BrainTumorLab" link

If the link does not open (browser blocks third-party cookies), show the app inside the notebook:

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)